# OpenAI Evidence Review

Prepare a small local OpenAI validation sample, then render Keynote 16:9 evidence figures that pair the extracted frame, nearby transcript text, and ABCD/Shorts feature checklist. The workflow validates extraction, cache, and visualization plumbing; it is not a brand classification accuracy review.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'notebooks' / '20260619_openai_evidence_review' / 'evidence_review.py').exists():
  NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebooks' / '20260619_openai_evidence_review'
if str(NOTEBOOK_DIR) not in sys.path:
  sys.path.insert(0, str(NOTEBOOK_DIR))

from evidence_review import (
    discover_sample_videos,
    build_validation_command,
    run_validation_sample,
    load_json,
    load_feature_rows,
    load_preprocess_manifest,
    select_review_frame,
    transcript_snippet,
    render_evidence_figure,
)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'main.py').exists():
  ROOT = ROOT.parents[1]

OUTPUT_DIR = ROOT / 'outputs' / 'openai_validation_sample'
ASSESSMENT_JSON = OUTPUT_DIR / 'assessment.json'
CACHE_DIR = ROOT / '.cache' / 'abcds-detector'
FIGURES_DIR = ROOT / 'notebooks' / '20260619_openai_evidence_review' / 'figures'

# Default output path: outputs/openai_validation_sample
# Default preprocessor cache path: .cache/abcds-detector
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
selected = discover_sample_videos(
    ROOT / 'sample_videos',
    per_platform=2,
    seed=20260619,
)
selected

In [ ]:
command = build_validation_command(selected, ASSESSMENT_JSON)
print(' '.join(str(part) for part in command))

In [ ]:
# Live validation is intentionally opt-in so opening the notebook never calls the API.
# run_validation_sample(command)

In [ ]:
if ASSESSMENT_JSON.exists():
  feature_rows_by_video = load_feature_rows(ASSESSMENT_JSON)
else:
  feature_rows_by_video = {}
  print(f'Run validation first; missing {ASSESSMENT_JSON}')

feature_rows_by_video

In [ ]:
# Render figures after validation and local preprocessing cache outputs exist.
rendered_figures = []

for video_uri, rows in feature_rows_by_video.items():
  video_key = Path(video_uri).stem
  try:
    manifest = load_preprocess_manifest(CACHE_DIR, video_uri)
    frame_path, timestamp_seconds = select_review_frame(
        manifest,
        prefer_first_5=True,
    )
  except (FileNotFoundError, ValueError) as error:
    print(f'Skipping {video_uri}; {error}')
    continue

  snippet = transcript_snippet(manifest, timestamp_seconds)
  output_path = FIGURES_DIR / f'{video_key}_evidence.png'

  rendered_figures.append(render_evidence_figure(
      frame_path=frame_path,
      transcript_text=snippet,
      feature_rows=rows,
      output_path=output_path,
      title=video_uri,
  ))

rendered_figures